In [ ]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from typing import TypedDict, Literal
from pydantic import Field, BaseModel

In [ ]:
load_dotenv()
model = ChatGroq(model="llama-3.3-70b-versatile")

In [ ]:
class SentimentSchema(BaseModel):
    sentiment: Literal["Positive", "Negative"] = Field(
        description="Sentiment of the review"
    )

In [ ]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(
        description="The category of issue mentioned in the review"
    )
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(
        description="The emotional tone expressed by the user"
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgent or critical the issue appears to be"
    )


In [ ]:
structured_model = model.with_structured_output(SentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)

In [ ]:
prompt = "What is the sentiment of the given reveiw. \n Phone heats a lot..."
print(structured_model.invoke(prompt))

In [ ]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["Positive", "Negative"]
    diagnosis: dict
    response: str

In [ ]:
def find_sentiment(state: ReviewState):
    review = state["review"]
    prompt = f"For the following review find out the sentiment. \n {review}"

    sentiment = structured_model.invoke(prompt).sentiment

    return {"sentiment": sentiment}


def check_sentiment(
        state: ReviewState,
) -> Literal["positive_response", "run_diagnosis"]:
    if state["sentiment"] == "Positive":
        return "positive_response"
    else:
        return "run_diagnosis"


def run_diagnosis(state: ReviewState):
    prompt = f"""Diagnose this negative review:\n\n{state["review"]}\n"
    "Return issue_type, tone, and urgency.
"""
    response = structured_model2.invoke(prompt)

    return {"diagnosis": response.model_dump()}


def negative_response(state: ReviewState):
    diagnosis = state["diagnosis"]

    prompt = f"""You are a support assistant.
The user had a '{diagnosis["issue_type"]}' issue, sounded '{diagnosis["tone"]}', and marked urgency as '{diagnosis["urgency"]}'.
Write an empathetic, helpful resolution message.
"""
    response = model.invoke(prompt).content

    return {"response": response}


def positive_response(state: ReviewState):
    prompt = f"""Write a warm thank-you message in response to this review:
    \n\n\"{state["review"]}\"\n
Also, kindly ask the user to leave feedback on our website."""

    response = model.invoke(prompt).content

    return {"response": response}


In [ ]:
graph = StateGraph(ReviewState)

graph.add_node("find_sentiment", find_sentiment)
graph.add_node("run_diagnosis", run_diagnosis)
graph.add_node("negative_response", negative_response)
graph.add_node("positive_response", positive_response)

graph.add_edge(START, "find_sentiment")
graph.add_conditional_edges("find_sentiment", check_sentiment)

graph.add_edge("positive_response", END)
graph.add_edge("run_diagnosis", "negative_response")
graph.add_edge("negative_response", END)

workflow = graph.compile()

In [ ]:
workflow

In [ ]:
initial_state = {"review": "Phone is excellent and i am loving it"}

response = workflow.invoke(initial_state)

print(response)